In [8]:
!cp "/content/drive/MyDrive/Hate Speech LLMs/Data ICWSM/Eval Data/o3Data.xlsx" .

In [9]:
import numpy as np
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score

In [10]:
df = pd.read_excel("o3Data.xlsx")

In [11]:
ref_col = 'Hop_2_Target_Targeted_Group (Ref)'
pred_col = 'Hop_2_Target_Targeted_Group'

In [12]:
df = df[df["Hop_1_Triage (Ref)"] != "بی‌ضرر"]
df = df[df["Hop_1_Triage"] != "بی‌ضرر"]
df[pred_col] = df[pred_col].fillna('[]')
print(len(df))

882


In [13]:
refGroups = []
for refGroup in df[ref_col]:
  refGroups.extend(ast.literal_eval(refGroup))
set(refGroups)

{'other',
 'دانشجویان، کودکان و نوجوانان',
 'دشمنان خارجی',
 'روحانیون و رهبران مذهبی',
 'زنان ایرانی',
 'شرکت\u200cها، سازمان\u200cها و کارکنان آن\u200cها',
 'فعالان حقوق بشر، رسانه\u200cای و دانشگاهی',
 'مخالفان سیاسی داخلی',
 'مردان ایرانی',
 'مقامات و سیاستمداران ایران',
 'مهاجران و اقلیت\u200cهای قومی/مذهبی',
 'نیروهای امنیتی و نظامی ایران',
 'هنرمندان، بازیگران و سلبریتی\u200cها'}

In [14]:
llmGroups = []
for llmGroup in df[pred_col]:
  llmGroups.extend(ast.literal_eval(llmGroup))
set(llmGroups)

{'other',
 'دانشجویان، کودکان و نوجوانان',
 'دشمنان خارجی',
 'روحانیون و رهبران مذهبی',
 'زنان ایرانی',
 'شرکت\u200cها، سازمان\u200cها و کارکنان آن\u200cها',
 'فعالان حقوق بشر، رسانه\u200cای و دانشگاهی',
 'مخالفان سیاسی داخلی',
 'مردان ایرانی',
 'مقامات و سیاستمداران ایران',
 'مهاجران و اقلیت\u200cهای قومی/مذهبی',
 'نیروهای امنیتی و نظامی ایران',
 'هنرمندان، بازیگران و سلبریتی\u200cها'}

In [15]:
len(set(refGroups))

13

In [16]:
len(set(llmGroups))

13

In [17]:
df[ref_col] = df[ref_col].apply(ast.literal_eval)
df[pred_col] = df[pred_col].apply(ast.literal_eval)

In [18]:
# Remove "other" from both columns
df[ref_col] = df[ref_col].apply(
    lambda x: [label for label in x if label != 'other']
)

df[pred_col] = df[pred_col].apply(
    lambda x: [label for label in x if label != 'other']
)

# Remove rows where either side becomes empty
df_eval = df[
    df[ref_col].apply(len).gt(0) &
    df[pred_col].apply(len).gt(0)
].copy()

In [19]:
mlb = MultiLabelBinarizer()

mlb.fit(list(df_eval[ref_col]) + list(df_eval[pred_col]))

y_true = mlb.transform(df_eval[ref_col])
y_pred = mlb.transform(df_eval[pred_col])

In [20]:
f1_micro = f1_score(y_true, y_pred, average='micro')
f1_macro = f1_score(y_true, y_pred, average='macro')

print(f"Micro F1: {f1_micro:.4f}")
print(f"Macro F1: {f1_macro:.4f}")

Micro F1: 0.8049
Macro F1: 0.7704
